# Email Spam Detection Machine Learning Project

This notebook demonstrates the complete end-to-end machine learning workflow for building an **Email Spam Detection Classifier** in Python. We will explore a dataset of emails, preprocess the text using Natural Language Processing (NLP) techniques, convert the text into numerical vectors using TF-IDF, train multiple classifiers (Naive Bayes, Logistic Regression, SVM), evaluate their performance, and save the final models.

## 1. Setup and Dependencies

First, we will import the required packages and configure the project path.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Add backend to system path to import preprocessor
sys.path.append(os.path.abspath(os.path.join("..")))
from backend.app.preprocess import preprocess_text, clean_text

print("Setup complete!")

## 2. Load the Dataset

We will load the downloaded dataset `spam_or_not_spam.csv` which has 3000 rows containing emails and their labels (1 = Spam, 0 = Ham/Not Spam).

In [ ]:
dataset_path = os.path.join("..", "dataset", "spam_or_not_spam.csv")
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Data Cleaning and Exploration

Let's look at label distribution, check for missing values, and handle duplicates.

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

# Handle missing values if any
df.dropna(inplace=True)

# Duplicate values
print(f"\nDuplicate rows found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)

print(f"\nShape after cleaning: {df.shape}")
print("\nLabel distribution:")
print(df['label'].value_counts())
df['label'].value_counts().plot(kind='bar', color=['#3b82f6', '#ef4444'])
plt.title("Label Distribution (0 = Ham, 1 = Spam)")
plt.xticks(rotation=0)
plt.show()

## 4. NLP Preprocessing

We will apply the preprocessing pipeline: lowercasing, HTML/URL/Email address removal, tokenization, stopword removal, and lemmatization.

In [ ]:
print("Preprocessing text samples...")
# Show a sample comparison before and after preprocessing
sample_idx = 10
sample_original = df.iloc[sample_idx]['email']
sample_preprocessed = preprocess_text(sample_original)

print("Original: ", sample_original[:300], "...")
print("-" * 50)
print("Preprocessed: ", sample_preprocessed[:300], "...")

# Apply to entire dataset
df['processed_text'] = df['email'].apply(preprocess_text)
df = df[df['processed_text'] != ""]
print(f"Final dataset shape: {df.shape}")

## 5. Feature Engineering: TF-IDF Vectorization

Next, we will split the dataset into train and test sets, and use the `TfidfVectorizer` to extract numerical features from our text data.

In [ ]:
X = df['processed_text']
y = df['label']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"TF-IDF Matrix dimensions for train set: {X_train_vectorized.shape}")

## 6. Model Building & Training

We will train three classification models: Naive Bayes, Logistic Regression, and Support Vector Machine (SVM).

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Support Vector Machine": SVC(kernel='linear', probability=True, random_state=42)
}

metrics_summary = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vectorized, y_train)
    y_pred = model.predict(X_test_vectorized)
    
    # Calculate Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    metrics_summary[name] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": cm.tolist()
    }
    
    print(f"[{name}] Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n")

## 7. Performance Visualization

Let's compare the performance of all three models side-by-side using charts.

In [ ]:
# Plot Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, (name, metrics) in enumerate(metrics_summary.items()):
    cm = np.array(metrics["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Not Spam', 'Spam'], 
                yticklabels=['Not Spam', 'Spam'], ax=axes[idx])
    axes[idx].set_title(f"Confusion Matrix: {name}")
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')
plt.tight_layout()
plt.show()

# Plot Metrics Comparison
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
nb_vals = [metrics_summary["Naive Bayes"]["accuracy"], metrics_summary["Naive Bayes"]["precision"], metrics_summary["Naive Bayes"]["recall"], metrics_summary["Naive Bayes"]["f1_score"]]
lr_vals = [metrics_summary["Logistic Regression"]["accuracy"], metrics_summary["Logistic Regression"]["precision"], metrics_summary["Logistic Regression"]["recall"], metrics_summary["Logistic Regression"]["f1_score"]]
svm_vals = [metrics_summary["Support Vector Machine"]["accuracy"], metrics_summary["Support Vector Machine"]["precision"], metrics_summary["Support Vector Machine"]["recall"], metrics_summary["Support Vector Machine"]["f1_score"]]

ax.bar(x - width, nb_vals, width, label='Naive Bayes', color='#3b82f6')
ax.bar(x, lr_vals, width, label='Logistic Regression', color='#10b981')
ax.bar(x + width, svm_vals, width, label='Support Vector Machine', color='#8b5cf6')

ax.set_ylabel('Scores')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0.8, 1.02)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 8. Save Models and Vectorizer

Finally, we will save the trained models, the TF-IDF Vectorizer, and the JSON metrics summary to the `model/` folder.

In [ ]:
os.makedirs(os.path.join("..", "model"), exist_ok=True)

# Save Models
for name, model in models.items():
    filename = os.path.join("..", "model", f"{name.lower().replace(' ', '_')}_model.pkl")
    joblib.dump(model, filename)
    print(f"Saved {name} model to {filename}")

# Save Vectorizer
joblib.dump(vectorizer, os.path.join("..", "model", "tfidf_vectorizer.pkl"))
print("Saved TF-IDF Vectorizer!")

# Save Metrics JSON
with open(os.path.join("..", "model", "metrics_comparison.json"), "w") as f:
    json.dump(metrics_summary, f, indent=4)
print("Saved metrics JSON summary!")